특징 선택(Feature Selection)과 차원 축소(Dimensionality Reduction)는 모두 모델의 성능을 높이고 과적합을 방지하기 위해 변수의 개수를 줄이는 기법이지만, **기존 변수를 다루는 방식**에서 근본적인 차이가 있습니다.

### 핵심 차이점 비교

| **구분** | **특징 선택 (Feature Selection)** | **차원 축소 (Feature Extraction)** |
| --- | --- | --- |
| **작업 방식** | 원본 변수 중 일부를 **선택하여 남기고, 나머지는 버림** | 기존 변수들을 조합/변환하여 **새로운 변수를 생성** |
| **데이터 변형** | 원본 데이터의 형태와 값을 **그대로 유지** | 데이터 원본이 축소된 공간으로 **투영/변환됨** |
| **해석 가능성** | **높음** (어떤 변수가 선택되었는지 직관적 확인 가능) | **낮음** (새로 조합된 변수가 의미하는 바를 알기 어려움) |
| **정보 손실** | 선택되지 않은 변수의 정보는 **완전히 삭제됨** | 기존 변수들의 정보량이 **상대적으로 많이 보존됨** |
| **주요 기법** | RFE, SelectKBest, Lasso(L1), ReliefF 등 | PCA, LDA, t-SNE, Autoencoder 등 |

### 이해를 돕는 직관적 비유

- **특징 선택:** 식료품 가방에서 요리에 꼭 필요한 야채(당근, 양파)만 **골라내고 나머지는 버리는 것**
    - 결과물: 당근, 양파 (원래 재료의 형태 유지)
- **차원 축소:** 여러 야채와 과일을 믹서기에 넣고 갈아서 **새로운 착즙 주스를 만드는 것**
    - 결과물: 주스 (기존 재료의 영양소/정보는 섞여 들어갔지만, 개별 형태는 사라짐)

### 상황별 선택 가이드

- **특징 선택을 써야 하는 경우**
    - "어떤 변수가 현상에 가장 중요한 영향을 미치는지" **원인 분석 및 비즈니스 해석이 필수적일 때**
    - 변수의 의미를 현업 담당자나 고객에게 설명해야 할 때
- **차원 축소를 써야 하는 경우**
    - 변수 간 상관관계가 매우 높은 **다공선성(Multicollinearity) 문제가 심할 때**
    - 개별 변수의 의미보다는 **전체적인 데이터 분포나 패턴 정보를 최대한 유지**하며 차원만 줄이고 싶을 때
    - 고차원 데이터를 2차원/3차원으로 **시각화**하고 싶을 때

In [ ]:
import os
import joblib
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. 데이터 로드 및 분할

cancer = load_breast_cancer()
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = cancer.target

X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.2, random_state=42, stratify=y
)

# 2. 필수 전처리: 표준화 스케일링

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'원본 변수 개수: {X.shape[1]}개\n')

# ==========================================
# [방법 A] 특징 선택 (Feature Selection: SelectKBest)
# ==========================================

selector = SelectKBest(score_func=f_classif, k=5)
X_train_fs = selector.fit_transform(X_train_scaled, y_train)
X_test_fs = selector.transform(X_test_scaled)

selected_features = X.columns[selector.get_support()]
print('--- [특징 선택] ---')
print(f'선택된 특징 (5개): {list(selected_features)}')

# max_iter=5000 추가하여 수렴 경고 해결

model_fs = LogisticRegression(
solver='saga', l1_ratio=0.0, max_iter=5000, random_state=42
)
model_fs.fit(X_train_fs, y_train)
y_pred_fs = model_fs.predict(X_test_fs)

print(f'정확도(Accuracy): {accuracy_score(y_test, y_pred_fs):.4f}')
print(classification_report(y_test, y_pred_fs))

# ==========================================
# [방법 B] 차원 축소 (Feature Extraction: PCA)
# ==========================================

pca = PCA(n_components=5, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print('--- [차원 축소] ---')
print(
'PCA 설명된 분산 비율 합계 (정보 보존량):'
f' {pca.explained_variance_ratio_.sum():.4f}'
)

# max_iter=5000 추가하여 수렴 경고 해결

model_pca = LogisticRegression(
solver='saga', l1_ratio=0.0, max_iter=5000, random_state=42
)
model_pca.fit(X_train_pca, y_train)
y_pred_pca = model_pca.predict(X_test_pca)

print(f'정확도(Accuracy): {accuracy_score(y_test, y_pred_pca):.4f}')
print(classification_report(y_test, y_pred_pca))

# ==========================================
# 폴더 확인 후 두 모델 저장
# ==========================================

save_dir = 'model'
if not os.path.exists(save_dir):
os.makedirs(save_dir)

path_fs = os.path.join(save_dir, 'cancer_feature_selection_model.pkl')
joblib.dump(model_fs, path_fs)

path_pca = os.path.join(save_dir, 'cancer_pca_model.pkl')
joblib.dump(model_pca, path_pca)

print('=' * 40)
print(f'특징 선택 모델 저장 완료: {path_fs}')
print(f'차원 축소 모델 저장 완료: {path_pca}')